In [1]:
import pandas as pd
import simplejson
import os
import re

In [ ]:
# Resolved from the repo root so this behaves the same in Jupyter (cwd = this directory)
# as under `./run.py language-to-json`.
repo = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(repo, 'src', 'assets', 'data')):
    repo, prev = os.path.dirname(repo), repo
    assert repo != prev, 'run this from inside the wingsearch checkout'

i18n_folder = os.path.join(repo, 'i18n')
result_folder = os.path.join(repo, 'src', 'assets', 'data', 'i18n')
files = [os.path.join(i18n_folder, file) for file in os.listdir(i18n_folder) if file.endswith('.xlsx') and not file.startswith('template')]

In [ ]:
# The sheets carry three columns that exist only so a translator can tell which row is which
# bird: `Expansion`, `English name` and `Scientific name`. `setLanguage` merges just four keys
# over the English card -- Common name, Power text, Flavor text, Note -- plus the four bonus-card
# flags, so none of the three is ever read at runtime.
#
# `Scientific name` is dropped with `Expansion`: it is a fifth of each file gzipped (nl 36KB ->
# 30KB) and a bird's latin name is the same in every language anyway. `English name` stays, at
# about 6KB, because it is what makes a diff of these generated files reviewable -- a translation
# is checked by reading it, and "607" alone does not tell a reviewer which bird changed.
identification = ['Expansion', 'Scientific name']


def filled(rows):
    """Drop the rows nobody has translated yet.

    sync-i18n-sheets.py gives every workbook a row for all 747 cards, so most sheets now carry
    hundreds of rows holding nothing but the English name that identifies the bird. The reducer
    merges field by field and skips a null, so such a row is indistinguishable from no row at
    all -- shipping them added about 850KB across the eleven files and changed nothing on screen.
    """
    return {i: row for i, row in rows.items()
            if any(v == v and str(v).strip() for key, v in row.items() if key != 'English name')}


for file in files:
    birds = pd.read_excel(file, sheet_name='Birds', index_col=0).drop(columns=identification).to_dict(orient='index')
    bonuses = pd.read_excel(file, sheet_name='Bonuses', index_col=0).drop(columns="Expansion").to_dict(orient='index')
    goals = pd.read_excel(file, sheet_name='Goals', index_col=0).drop(columns="Expansion").to_dict(orient='index')
    other = pd.read_excel(file, sheet_name='Other', index_col=0).to_dict(orient='index')
    parameters = pd.read_excel(file, sheet_name='Parameters', index_col=0).to_dict(orient='index')

    result = {'birds': filled(birds), 'bonuses': filled(bonuses), 'goals': filled(goals),
              'other': other, 'parameters': parameters}
    json_file = re.search(r'[\w]+\.xlsx', file).group().replace('.xlsx', '.json')

    if not os.path.exists(result_folder):
        os.makedirs(result_folder)

    # An explicit encoding and newline, so this writes the same bytes on every platform. The
    # committed files were generated on Windows, where text mode turns every \n into \r\n: any
    # regeneration elsewhere rewrote all eleven files with nothing changed in them, which buried
    # whatever had actually changed under 66,000 changed lines.
    with open(os.path.join(result_folder, json_file), 'w', encoding='utf-8', newline='\n') as fp:
        simplejson.dump(result, fp, ignore_nan=True, indent=2)
